In [6]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

load_dotenv()

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

In [7]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print('Connected')

Connected


In [8]:
graph.refresh_schema()

print(graph.schema)

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER, status: STRING}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [9]:
query = """
MATCH (student:Student) - [:ENROLLED_IN] -> (course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id
ORDER BY student_id, course_id
"""

result = graph.query(query)
result

[{'student_name': '홍길동',
  'course_name': 'Python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data Analysis',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': 'Database',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': 'Data Analysis',
  'student_id': 2,
  'course_id': 104},
 {'student_name': '이민수',
  'course_name': 'Machine Learning',
  'student_id': 3,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'Langchain',
  'student_id': 3,
  'course_id': 106},
 {'student_name': '박서연',
  'course_name': 'Machine Learning',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': 'Deep Learning',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': 'Database',
  'student_id': 5,
  'course_id': 102},
 {'student_name': '최준호',
  'course_name': 'Langchain',
  'student_id': 5,
  'course_id': 106}]

In [10]:
llm = ChatOpenAI(
    model=os.getenv('OPENAI_MODEL'),
    temperature=0   # DB 정보 기반으로 가져올 것이므로 창의성 0
)

In [11]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    validate_cypher=True,
    return_intermediate_steps=True,
    top_k=10,
    allow_dangerous_requests=True
)

In [12]:
question = '파이썬 수강생'

response = chain.invoke({'query': question})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
WHERE c.name CONTAINS '파이썬' OR c.name CONTAINS 'Python'
RETURN s;
Full Context:
[{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}}]

> Finished chain.


In [13]:
response['result']

'홍길동입니다.'

In [ ]:
print(response.get('intermediate'))

None


In [16]:
def ask_graph(question: str) -> str:
    if not question.strip():
        raise ValueError('질문 입력 필요')

    response = chain.invoke({'query': question})

    print(f'[질문] {question}')
    print(f'[답변] {response['result']}')

    for step in response.get('intermediate_steps', []):
        if 'query' in step:
            print(f"[생성된 Cypher] {step['query']}")
        if 'context' in step:
            print(f"[조회 결과] {step['context']}")

    return response

In [19]:
ask_graph('Tell me about the lectures 홍길동 took and the instructor in charge')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS lecture, i.name AS instructor;
Full Context:
[{'lecture': 'Python', 'instructor': 'Capybara'}, {'lecture': 'Data Analysis', 'instructor': 'Alice'}]

> Finished chain.
[질문] Tell me about the lectures 홍길동 took and the instructor in charge
[답변] 홍길동이 수강한 강의는 Python과 Data Analysis이며, 담당 강사는 각각 Capybara와 Alice입니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS lecture, i.name AS instructor;
[조회 결과] [{'lecture': 'Python', 'instructor': 'Capybara'}, {'lecture': 'Data Analysis', 'instructor': 'Alice'}]


{'query': 'Tell me about the lectures 홍길동 took and the instructor in charge',
 'result': '홍길동이 수강한 강의는 Python과 Data Analysis이며, 담당 강사는 각각 Capybara와 Alice입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN c.name AS lecture, i.name AS instructor;"},
  {'context': [{'lecture': 'Python', 'instructor': 'Capybara'},
    {'lecture': 'Data Analysis', 'instructor': 'Alice'}]}]}

In [20]:
ask_graph('수강생 가장 많은 강의')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
Full Context:
[{'c': {'duration': 36, 'course_id': 102, 'level': '중급', 'name': 'Database'}, 'student_count': 2}]

> Finished chain.
[질문] 수강생 가장 많은 강의
[답변] 수강생이 가장 많은 강의는 **Database**이며, 수강생은 **2명**입니다.
[생성된 Cypher] MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
[조회 결과] [{'c': {'duration': 36, 'course_id': 102, 'level': '중급', 'name': 'Database'}, 'student_count': 2}]


{'query': '수강생 가장 많은 강의',
 'result': '수강생이 가장 많은 강의는 **Database**이며, 수강생은 **2명**입니다.',
 'intermediate_steps': [{'query': 'MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nRETURN c, count(s) AS student_count\nORDER BY student_count DESC\nLIMIT 1'},
  {'context': [{'c': {'duration': 36,
      'course_id': 102,
      'level': '중급',
      'name': 'Database'},
     'student_count': 2}]}]}